In [2]:
#IAAA
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import math

print("Libraries loaded.")
print("MPS available:", torch.backends.mps.is_available())

Libraries loaded.
MPS available: True


In [3]:
class IAAAttention(nn.Module):
    def __init__(self, d_model, nhead):
        super(IAAAttention, self).__init__()
        
        self.d_model = d_model
        self.nhead = nhead
        self.d_head = d_model // nhead
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        # Learnable alpha — the IAA contribution
        self.alpha = nn.Parameter(torch.tensor(1.0))
    
    def forward(self, x, class_bias):
        batch_size, seq_len, _ = x.shape
        
        # Project input to Q, K, V
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Reshape for multi-head attention
        Q = Q.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        
        # Standard attention scores: QK^T / sqrt(dk)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_head)
        
        # IAA modification: add alpha * log(1/f_c)
        # class_bias shape: [batch_size] -> [batch_size, 1, 1, 1]
        bias = self.alpha * class_bias.view(batch_size, 1, 1, 1)
        scores = scores + bias
        
        # Softmax and weighted sum
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)
        
        # Reshape back
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(output)
        
        return output